In [9]:
"""
chute_drop_batch.py

Batch Monte-Carlo chute-drop validation harness.

Reads the geometric pose analysis produced by ChutePoseAnalysis.m (MATLAB)
for one or more parts -- specifically each condition's
"<folder>_MASTER_summary.txt" -- and, for every (part, roll, pitch)
combination requested, runs N randomized PyBullet drop trials into a chute
tilted to that same roll/pitch. Each trial's settled orientation is matched
(in the chute's LEVEL/untilted local frame, exactly as ChutePoseAnalysis.m
assumes) against the refQuat catalog for that condition, and match counts
are tallied into an empirical pose-frequency distribution that can be
compared directly against the CSA_B / CSA_A / CSA_N / CRSA_B / CRSA_A /
CRSA_N columns already computed by the MATLAB analysis.

--------------------------------------------------------------------------
EXPECTED DIRECTORY LAYOUT (as produced by ChutePoseAnalysis.m)
--------------------------------------------------------------------------
    <parts_folder>/
        PartA.stl
        PartB.stl
        results/
            PartA/
                PartA_15R_15P/
                    PartA_15R_15P_MASTER_summary.txt
                    PartA_15R_15P_GEOMETRIC_summary.txt
                    PartA_15R_15P.pdf
                    PartA_15R_15P_poseCatalog.csv      <- OPTIONAL, see below
                PartA_15R_25P/
                    ...
            PartB/
                ...

--------------------------------------------------------------------------
POSE CATALOG (symmetric-variant matching)
--------------------------------------------------------------------------
_MASTER_summary.txt stores metadata (CSA/CRSA weights, thresholds,
ratioWall/ratioFloor, TRANSITIONS/FLOOR-UNSTABLE notes) per STABLE pose,
merged by mergeByCSAValues -- that merge groups raw pose indices whose CSA
weight matched, and its refQuat field is not guaranteed to carry every
symmetric-orientation variant that got folded into the merge (only their
count, "nMerge"). Matching a dropped part's final orientation against a
single representative quaternion under-matches any pose that has
rotational symmetry (e.g. a square face has 4 equivalent resting
orientations 90 deg apart in quaternion space -- far outside
quatMatchTol).

This script builds the actual quaternion-matching catalog from three
possible sources, in priority order:

  1. An external CSV next to the MASTER_summary: "<folder>_poseCatalog.csv"
     (or anything matching "*catalog*.csv" in that folder), with columns
     pose_id,qw,qx,qy,qz and one row per symmetric variant (this is the
     "exportPoseQuatCatalog.m" catalog referenced in the original
     chute_drop_sim.py). Preferred when present since it's the
     authoritative, unambiguous variant list.

  2. "<folder>_GEOMETRIC_summary.txt" -- this file groups poses by the
     EARLIER, pre-stability THETA merge (not the same grouping as
     MASTER's CSA-weight merge) and its Pose/refQuat columns can list
     multiple raw pose indices with one pipe-separated quaternion per
     index on the same row, e.g. Pose "1,3" with refQuat
     "[q1] | [q2]" -> raw index 1 has quaternion q1, raw index 3 has
     quaternion q2. This script parses that into a flat
     raw_pose_index -> quat lookup, then for every MASTER pose looks up
     each of its own merged raw indices (MASTER's si field, e.g. "5,15")
     in that lookup and stacks whatever's found as that pose's full
     symmetric-variant catalog entry. Used whenever no external
     poseCatalog.csv is present.

  3. If neither of the above is found, falls back to whatever
     quaternion(s) MASTER_summary's OWN refQuat field parsed for that row
     (which may itself list multiple pipe-separated quats if that file
     was also given the multi-quaternion format -- see
     _parse_quat_list/POSE_ROW_RE below -- or just one, the old format).

Regardless of source, if a pose's final variant count still falls short
of its nMerge count, a targeted warning names exactly which pose(s) are
still under-matched rather than a blanket "no catalog" warning.

--------------------------------------------------------------------------
ANGLE GRID
--------------------------------------------------------------------------
    alphas (roll)  = 15, 17.5, 20   deg
    betas  (pitch) = 15, 25, 35, 45 deg

The chute is physically tilted using the EXACT alpha/beta values above.
The results-folder name is looked up using MATLAB's round() convention
(round-half-away-from-zero), matching ChutePoseAnalysis.m's
`sprintf('%s_%dR_%dP', partName, round(chuteRoll_deg), round(chutePitch_deg))`
-- so alpha=17.5 physically tilts the chute 17.5 deg but looks up folder
tag "18R". This mirrors the MATLAB code's own math/display discrepancy.

--------------------------------------------------------------------------
CONTACT MODEL -- PTFE chute (perforated, textured) / PTFE-coated part
--------------------------------------------------------------------------
Both bodies are nominally near-frictionless PTFE-on-PTFE (mu ~ 0.04-0.15),
but the chute is perforated and surface-textured, so the *effective*
contact friction it presents is not a single constant -- hole rims and
rough patches can grip noticeably more than the bulk PTFE surface, and a
part edge can briefly catch on a hole. Rather than modelling hole geometry
explicitly, this script:
  - redraws chute and part friction/restitution from configurable bands
    EVERY trial (--chute-friction-*, --part-friction-*, --*-restitution-*)
  - with probability --snag-prob, additionally boosts that trial's chute
    friction by --snag-friction-boost, representing an edge catching on a
    hole rim/burr
  - sets low rolling/spinning friction for both bodies (PTFE has very low
    rolling resistance), via --rolling-friction / --spinning-friction
  - optionally (--roughness-enabled) applies a tunable, contact-conditioned
    micro-roughness force/torque WHILE the part is sliding, simulating
    stick-slip skitter across surface texture without modelling the
    texture geometrically -- see RoughnessModel / apply_roughness_step
    below for the full explanation.

This is a coarse statistical stand-in for hole/texture geometry, not a
geometric model of the perforations -- flagged here explicitly since it's
a simplification.

--------------------------------------------------------------------------
SURFACE ROUGHNESS MODEL (--roughness-enabled)
--------------------------------------------------------------------------
The friction/restitution redraw above captures *between-trial* variation
(the part happens to land on a rougher or smoother patch this drop). It
does NOT capture *within-trial* roughness -- a part sliding down a real
textured/perforated chute doesn't glide on a perfect mathematical plane,
it skitters: it catches a bump, deflects slightly, catches another a few
mm later, etc. The old --no-ambient-jitter / ambient_jitter mechanism is
NOT that -- it fires a single big kick on a fixed step-count timer purely
to shake the part out of numerically-stable-but-physically-impossible
false settles. It is a settle-detector guard, not a texture model, and it
still runs independently of roughness.

RoughnessModel adds a second, purpose-built noise process:
  - Every `--roughness-correlation-steps` simulation steps, a new random
    lateral force direction + random-axis torque ("one bump") is sampled
    and held constant, instead of resampling every single step -- this
    gives the noise some spatial correlation length (crossing one bump
    takes several steps, not one), rather than instantaneous white noise.
  - The force/torque amplitude is scaled by the part's current speed
    (ramped up to full strength by --roughness-velocity-scale-ref m/s,
    and gated off entirely below --roughness-velocity-gate m/s) so a part
    that has genuinely come to rest doesn't get perpetually buzzed awake
    by texture noise -- only a part that is actively sliding/rolling
    feels the roughness, which is physically what "rolling over bumps"
    means and keeps this from fighting run_until_settled's convergence.
  - By default (--roughness-contact-only, on unless disabled) the
    perturbation is only applied while the part is actually touching the
    chute (p.getContactPoints), so free-fall trajectories and the
    --drop-y-min/--drop-y-max entry-position dispersion are undisturbed.

This is deliberately a time-correlated random-force process, not a
geometric bump map -- it's a cheap, fully-tunable stand-in for "rough
surface" that doesn't require modelling the chute's actual perforation
pattern. --roughness-force-frac / --roughness-torque-frac (fractions of
the part's own weight) are the main knobs; raise them for a grippier /
more textured surface, lower for a smoother one. It applies during BOTH
the initial settle and the knife-edge-recheck settle after the nudge
perturbation, since a real rough surface would still be textured during
that recheck too.

--------------------------------------------------------------------------
STABILITY THRESHOLD CROSS-REFERENCE
--------------------------------------------------------------------------
Every condition's THRESH_WALL / THRESH_FLOOR (parsed from the
MASTER_summary header) is carried into the output, along with each
matched pose's ratioWall/ratioFloor and MATLAB's own
TRANSITIONS / FLOOR-UNSTABLE / Q=0 notes, so simulated hits on a pose the
geometric analysis flagged as invalid/unstable are visible in the summary
rather than silently folded into the count.

--------------------------------------------------------------------------
UNITS -- PyBullet is SI (meters). Your mesh files almost certainly are not.
--------------------------------------------------------------------------
STL/OBJ files store raw numbers with no embedded unit. CAD tools very
commonly export STL in millimeters. PyBullet has no way to know this and
will treat every coordinate as meters -- so an 80x69x15 "mm" part becomes
an 80x69x15 METER object (roughly the size of a building) unless you tell
it otherwise via --part-scale / --chute-scale.

This script now runs an automatic mesh-scale sanity check (see
`describe_mesh_bounds` / `run_scale_sanity_check`) BEFORE simulating,
printing each mesh's real-world size in meters after scaling is applied,
and warning loudly if the part and chute end up wildly mismatched in size
(a near-certain sign one of --part-scale/--chute-scale is wrong). This
check runs even in --dry-run and even without --gui, since it's the
fastest way to catch a units bug without ever opening the visualizer.

--------------------------------------------------------------------------
CHUTE VERTICAL PLACEMENT (--chute-z-lift)
--------------------------------------------------------------------------
The chute mesh's local origin (0,0,0) sits ON the floor/wall seam (local
Z=0), not below the chute. reset_condition_world() places the chute at a
world position and then ROTATES it in place by the roll/pitch quaternion.
Rotating a flat floor about a pivot that lies on that same floor swings
part of the floor to the OPPOSITE side of the pivot's height once tilted
-- i.e. part of the chute dips below world Z=0 purely from the tilt, even
at moderate angles. Fix: the chute (and the drop point, which is defined
relative to the chute's local frame) are both lifted by --chute-z-lift
meters (world +Z) before any rotation/placement math, so the tilted mesh
never gets anywhere near the Z=0 ground plane regardless of roll/pitch.
final_x/final_y/final_z reported in the trials CSV have the lift
subtracted back out.

--------------------------------------------------------------------------
GUI CAMERA
--------------------------------------------------------------------------
When --gui is passed, the debug camera is automatically framed around the
chute's actual (post-scale, post-lift) bounding box every time the world
is rebuilt for a condition.

--------------------------------------------------------------------------
COLLISION ROBUSTNESS (CCD, solver iterations, false-settle guard)
--------------------------------------------------------------------------
Small, light, fast-moving parts are a known trouble spot for Bullet's
default discrete collision detection. This script enables CCD, raises
numSubSteps/numSolverIterations, and re-checks every "settled" pose with a
small perturbation impulse before accepting it, to catch
numerically-stable-but-physically-impossible poses.

--------------------------------------------------------------------------
COLLISION SHAPE FLAGS -- concave internal edges
--------------------------------------------------------------------------
The chute is a single concave triangle-mesh collision shape
(GEOM_FORCE_CONCAVE_TRIMESH). p.GEOM_CONCAVE_INTERNAL_EDGE tells Bullet's
internal-edge utility to smooth edge-adjacent contact normals so a part
sliding across a triangle seam doesn't pick up a spurious bounce. Must be
combined with GEOM_FORCE_CONCAVE_TRIMESH (bitwise OR).

--------------------------------------------------------------------------
TIMESTEP vs. SUBSTEPS -- do not confuse the two
--------------------------------------------------------------------------
p.setTimeStep(dt) sets simulated time per stepSimulation() call.
numSubSteps subdivides that call internally for accuracy -- it does NOT
add more simulated time. dt is kept at a conventional 1/240 s.

--------------------------------------------------------------------------
DROP POSITION -- fixed vs. randomized Y
--------------------------------------------------------------------------
--drop-xy X Y sets the drop offset in the chute's local frame (X = along
slide direction, Y = along the floor-recede axis), and --drop-height sets
the offset along the chute's local +Z. By default this offset is FIXED
for every trial in a condition.

If you also pass --drop-y-min and --drop-y-max (both required together),
the Y component is instead drawn fresh from Uniform(drop_y_min,
drop_y_max) on EVERY trial, while X (--drop-xy[0]) and --drop-height stay
fixed. This is useful for modeling dispersion in where parts actually
enter the chute rather than assuming every part lands at exactly the same
spot. The drop position (drop_x, drop_y, drop_height) actually used is
recorded per-trial in the _trials.csv output regardless of whether
randomization is active, so you can always see what was sampled.

--------------------------------------------------------------------------
USAGE
--------------------------------------------------------------------------
    python chute_drop_batch.py \
        --parts-folder /path/to/partsFolder \
        --chute chute.obj --chute-concave \
        --part-scale 0.001 \
        --n 1000 \
        --out-dir batch_drop_results

    # randomize where parts enter the chute (Y dispersion):
    python chute_drop_batch.py \
        --parts-folder /path/to/partsFolder \
        --chute chute.obj --chute-concave \
        --drop-xy -0.75 -2.0 \
        --drop-y-min -2.25 --drop-y-max -1.75 \
        --n 1000 \
        --out-dir batch_drop_results

    # tune-able rough-surface skitter, no chute geometry changes required:
    python chute_drop_batch.py \
        --parts-folder /path/to/partsFolder \
        --chute chute.obj --chute-concave \
        --part-scale 0.001 \
        --roughness-enabled \
        --roughness-force-frac 0.02 --roughness-torque-frac 0.02 \
        --roughness-correlation-steps 8 \
        --n 1000 \
        --out-dir batch_drop_results

    # just see what conditions would run, without simulating:
    python chute_drop_batch.py --parts-folder ... --chute chute.obj --dry-run

--------------------------------------------------------------------------
RUNNING FROM A JUPYTER NOTEBOOK
--------------------------------------------------------------------------
argparse normally reads from sys.argv, which inside a notebook kernel
contains Jupyter's own launcher arguments (e.g. ipykernel_launcher.py
-f <connection file>), not your CLI flags. To run this from a notebook
cell without hitting "the following arguments are required" errors, set
NOTEBOOK_ARGV *before* calling main() -- main() will detect it's running
under a kernel and use that list instead of sys.argv:

    import chute_drop_batch as cdb
    cdb.NOTEBOOK_ARGV = [
        "--parts-folder", r"C:\\path\\to\\partsFolder",
        "--chute", r"C:\\path\\to\\chute.obj",
        "--chute-concave",
        "--part-scale", "0.001",
        "--drop-xy", "-0.75", "-2.0",
        "--drop-y-min", "-2.25",
        "--drop-y-max", "-1.75",
        "--n", "1000",
        "--out-dir", "batch_drop_results",
    ]
    cdb.main()

    NOTE: every element of this list needs its own trailing comma -- two
    adjacent string literals with a missing comma between them get
    silently concatenated by Python (e.g. "--chute-concave" "--dry-run"
    becomes the single token "--chute-concave--dry-run"), which argparse
    will reject as an unrecognized argument.
"""

import argparse
import csv
import glob
import os
import re
import sys
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pybullet as p
import pybullet_data


# When running inside a Jupyter/IPython kernel, set this list of CLI-style
# args before calling main() -- see "RUNNING FROM A JUPYTER NOTEBOOK" above.
# Left as None when running normally from the command line.
NOTEBOOK_ARGV: Optional[List[str]] = None


def _running_under_notebook_kernel() -> bool:
    """Detect ipykernel/Jupyter so we know not to trust sys.argv."""
    if sys.argv and "ipykernel_launcher" in sys.argv[0]:
        return True
    return "ipykernel" in sys.modules


# ===========================================================================
# Quaternion primitives (kept in [w,x,y,z] order to match MATLAB exactly,
# converted to PyBullet's [x,y,z,w] only at the pybullet API boundary)
# ===========================================================================

def q_from_axis_angle_wxyz(axis, angle):
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    s = np.sin(angle / 2.0)
    return np.array([np.cos(angle / 2.0), s * axis[0], s * axis[1], s * axis[2]])


def q_compose_wxyz(q1, q2):
    """Matches ChutePoseAnalysis.m's q_compose(q1, q2) exactly."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w2 * w1 - x2 * x1 - y2 * y1 - z2 * z1,
        w2 * x1 + x2 * w1 + y2 * z1 - z2 * y1,
        w2 * y1 - x2 * z1 + y2 * w1 + z2 * x1,
        w2 * z1 + x2 * y1 - y2 * x1 + z2 * w1,
    ])


def wxyz_to_xyzw(q_wxyz):
    return (float(q_wxyz[1]), float(q_wxyz[2]), float(q_wxyz[3]), float(q_wxyz[0]))


def q_geodesic_wxyz(q1_wxyz, q2_wxyz):
    """Sign-invariant geodesic distance, matches MATLAB's q_geodesic."""
    dp = abs(np.dot(q1_wxyz, q2_wxyz))
    dp = min(dp, 1.0)
    return 2.0 * np.arccos(dp)


def random_quaternion_xyzw(rng: np.random.Generator):
    """Uniformly random unit quaternion (Shoemake's method) -> (x,y,z,w)."""
    u1, u2, u3 = rng.random(3)
    q_w = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    q_x = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    q_y = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    q_z = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return (q_x, q_y, q_z, q_w)


def build_chute_quat_xyzw(roll_deg: float, pitch_deg: float):
    """Reproduces ChutePoseAnalysis.m's chute frame exactly:
        roll  = -deg2rad(chuteRoll_deg)   about X
        pitch =  deg2rad(chutePitch_deg)  about Y
        q_chute = q_compose(q_roll, q_pitch)
    Returned in PyBullet (x,y,z,w) order.
    """
    roll = -np.deg2rad(roll_deg)
    pitch = np.deg2rad(pitch_deg)
    q_roll = q_from_axis_angle_wxyz([1, 0, 0], roll)
    q_pitch = q_from_axis_angle_wxyz([0, 1, 0], pitch)
    q_chute = q_compose_wxyz(q_roll, q_pitch)
    q_chute = q_chute / np.linalg.norm(q_chute)
    return wxyz_to_xyzw(q_chute)


def rotate_to_chute_local(final_quat_xyzw, chute_quat_xyzw):
    """Express a world-frame orientation in the chute's local (level) frame:
    q_local = q_chute^-1 * q_final. Required because refQuats in the
    MATLAB catalog assume a level chute (no Rchute tilt baked in).
    """
    q_chute_inv = p.invertTransform([0, 0, 0], chute_quat_xyzw)[1]
    _, q_local = p.multiplyTransforms([0, 0, 0], q_chute_inv, [0, 0, 0], final_quat_xyzw)
    return q_local


def rotate_vec_by_quat_xyzw(vec, quat_xyzw):
    pos, _ = p.multiplyTransforms([0, 0, 0], quat_xyzw, vec, [0, 0, 0, 1])
    return pos


def matlab_round(x: float) -> int:
    """Round-half-away-from-zero, matching MATLAB's round()."""
    return int(np.floor(abs(x) + 0.5) * (1 if x >= 0 else -1))


# ===========================================================================
# Pose catalog (symmetric variants) -- from exportPoseQuatCatalog.m CSV
# ===========================================================================

def load_pose_catalog(csv_path: str) -> Dict[int, np.ndarray]:
    """pose_id -> Nx4 array of quaternions in [w,x,y,z] order."""
    catalog: Dict[int, list] = {}
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pose_id = int(row["pose_id"])
            q_wxyz = [float(row["qw"]), float(row["qx"]), float(row["qy"]), float(row["qz"])]
            catalog.setdefault(pose_id, []).append(q_wxyz)
    return {k: np.array(v) for k, v in catalog.items()}


def find_catalog_csv(condition_dir: str, folder_name: str) -> Optional[str]:
    candidates = [
        os.path.join(condition_dir, f"{folder_name}_poseCatalog.csv"),
        os.path.join(condition_dir, f"{folder_name}_pose_catalog.csv"),
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c
    matches = sorted(glob.glob(os.path.join(condition_dir, "*catalog*.csv")))
    return matches[0] if matches else None


# ===========================================================================
# GEOMETRIC_summary.txt parser -- symmetric-variant quaternions, keyed by
# RAW (pre-CSA-merge) pose index
# ===========================================================================
#
# IMPORTANT: the multi-quaternion / comma-grouped format described above
# was added to *_GEOMETRIC_summary.txt ONLY -- *_MASTER_summary.txt keeps
# its own (possibly different) pose grouping from mergeByCSAValues, and is
# still the sole source for CSA/CRSA weights, thresholds, ratioWall/
# ratioFloor, and TRANSITIONS/FLOOR-UNSTABLE notes (parse_master_summary,
# above, is unchanged for those purposes).
#
# The two files group poses differently:
#   - GEOMETRIC_summary.txt groups by the pre-stability THETA merge (each
#     row here is 1-2 raw pose indices that share floor/wall contact sets
#     up to a theta rotation), e.g. "1,3" -> raw indices 1 and 3.
#   - MASTER_summary.txt groups by the (separate, post-stability) CSA-
#     weight merge (mergeByCSAValues), e.g. a MASTER row's si field "5,15"
#     -> raw indices 5 and 15 were folded into one stable pose because
#     they scored the same CSA/CRSA weight -- this may or may not be the
#     same grouping as GEOMETRIC's.
#
# To get the FULL set of symmetric-variant quaternions for a MASTER pose,
# we therefore parse GEOMETRIC_summary.txt into a flat raw_index -> quat
# lookup (parse_geometric_summary), then for each MASTER PoseEntry look up
# every one of its merged_ids (raw indices) in that flat table -- see the
# catalog-construction block in discover_conditions().

GEOM_ROW_RE = re.compile(
    r'^\s*(?P<pose>\d+(?:,\d+)*)\s+(?P<plane>\d+(?:,\d+)*)\s+'
    r'(?P<theta>-?\d+(?:\.\d+)?(?:,-?\d+(?:\.\d+)?)*)\s+'
    r'(?P<quats>\[[^\]]*\](?:\s*\|\s*\[[^\]]*\])*)'
    r'(?:\s+.*)?$'
)


def find_geometric_summary(condition_dir: str, folder_name: str) -> Optional[str]:
    candidate = os.path.join(condition_dir, f"{folder_name}_GEOMETRIC_summary.txt")
    if os.path.isfile(candidate):
        return candidate
    matches = sorted(glob.glob(os.path.join(condition_dir, "*GEOMETRIC*summary*.txt")))
    return matches[0] if matches else None


def parse_geometric_summary(geom_path: str) -> Tuple[Dict[int, np.ndarray], Dict[int, List[int]]]:
    """Parses a *_GEOMETRIC_summary.txt file. Returns a pair:

      raw_quats: flat raw_pose_index -> (4,) quat [w,x,y,z], by zipping
        each row's comma-separated Pose-index group with its
        pipe-separated refQuat bracket group, in order (e.g. Pose "1,3"
        with refQuat "[q1] | [q2]" -> {1: q1, 3: q2}).

      raw_group: raw_pose_index -> the FULL list of raw pose indices that
        appeared together on that same row (i.e. its complete theta-merge
        symmetric group, including itself), e.g. both raw index 1 and raw
        index 3 map to [1, 3].

    raw_group matters because MASTER_summary.txt may still reference only
    ONE representative raw index per pose (its old, unmodified format --
    see the module docstring's POSE CATALOG section) even though that raw
    index's geometric theta-group actually has more than one symmetric
    variant. Looking up a MASTER-referenced raw id's FULL group (rather
    than just that exact id) is what recovers those extra variants --
    see the catalog-construction block in discover_conditions().

    Rows that don't match the expected pattern (title/header/separator/
    "Column key" lines) are silently skipped, matching parse_master_summary's
    behavior for its own non-pose-row lines.
    """
    raw_quats: Dict[int, np.ndarray] = {}
    raw_group: Dict[int, List[int]] = {}
    with open(geom_path, "r") as f:
        for line in f:
            m = GEOM_ROW_RE.match(line)
            if not m:
                continue
            g = m.groupdict()
            pose_ids = [int(s) for s in g["pose"].split(",")]
            try:
                quats = _parse_quat_list(g["quats"])
            except ValueError:
                continue
            if len(quats) != len(pose_ids):
                print(f"  [warn] {os.path.basename(geom_path)}: pose group "
                      f"'{g['pose']}' lists {len(pose_ids)} pose index/indices but "
                      f"{len(quats)} quaternion(s) in refQuat -- zipping what's "
                      f"available in order; any extra index will have no geometric "
                      f"quat.")
            for pid, q in zip(pose_ids, quats):
                raw_quats[pid] = q
            for pid in pose_ids:
                raw_group[pid] = pose_ids
    return raw_quats, raw_group


def match_pose_in_catalog(final_quat_xyzw, catalog: Dict[int, np.ndarray], tol: float):
    x, y, z, w = final_quat_xyzw
    q_wxyz = np.array([w, x, y, z])
    best_id, best_d = None, np.inf
    for pose_id, variants in catalog.items():
        for q_ref in variants:
            d = q_geodesic_wxyz(q_wxyz, q_ref)
            if d < best_d:
                best_d, best_id = d, pose_id
    if best_id is not None and best_d <= tol:
        return best_id, best_d
    return None, best_d

def _dedupe_quats(quats: List[np.ndarray], tol: float) -> np.ndarray:
    """Sign-invariant dedup of quaternion variants (q and -q are the same
    rotation). Used when merging MASTER_summary's own parsed refQuat(s)
    with GEOMETRIC_summary-derived variants, so the same underlying
    orientation found via two different sources isn't double-counted in
    catalog[si].shape[0] (which feeds the nMerge under-matched check).
    `tol` should be <= the match-time quat_match_tol, or two genuinely
    distinct close variants could get collapsed into one.
    """
    kept: List[np.ndarray] = []
    for q in quats:
        if not any(q_geodesic_wxyz(q, k) <= tol for k in kept):
            kept.append(q)
    return np.array(kept)

# ===========================================================================
# MASTER_summary.txt parser (metadata: CSA/CRSA weights, thresholds, notes)
# ===========================================================================
#
# NOTE ON MULTI-QUATERNION refQuat FIELDS (symmetric-variant indexing):
# The pose-index ("si") and refQuat columns can now each list MULTIPLE
# values on a single row -- si as a comma-separated group of merged pose
# indices (e.g. "1,3"), and refQuat as a pipe-separated list of one
# "[w x y z]" bracket per member of that group (e.g.
# "[0.7071 0.0000 -0.7071 0.0000] | [0.0000 0.7071 -0.0000 0.7071]"),
# matching the convention used in the GEOMETRIC_summary output. This
# replaces the old "exactly one refQuat per merged pose" limitation --
# previously the ONLY way to recover the other symmetric-orientation
# quaternions was a companion poseCatalog.csv (see find_catalog_csv /
# load_pose_catalog below). The regex and parser here accept BOTH the old
# single-si/single-quat format and the new comma/pipe multi-value format
# transparently: si is captured as a comma-separated group (a length-1
# group covers the old format) and refQuat is captured as one-or-more
# pipe-separated brackets (parsed by _parse_quat_list). The FIRST id in
# the si group is used as the canonical pose_id dict key; ALL ids in the
# group are kept on PoseEntry.merged_ids and ALL parsed quaternions are
# kept on PoseEntry.quat_wxyz as a (k,4) array, so downstream catalog
# construction (see discover_conditions) can use every symmetric variant
# even when no separate poseCatalog.csv is present.

POSE_ROW_RE = re.compile(
    r'^\s*(?P<si>\d+(?:,\d+)*)\s+(?P<plane>\d+(?:,\d+)*)\s+'
    r'(?P<theta>-?\d+(?:\.\d+)?(?:,-?\d+(?:\.\d+)?)*)\s+'
    r'(?P<quats>\[[^\]]*\](?:\s*\|\s*\[[^\]]*\])*)\s+'
    r'(?P<omega>-?\d+\.\d+)\s+(?P<height>-?\d+\.\d+)\s+'
    r'(?P<rwall>-?\d+\.\d+)\s+(?P<rfloor>-?\d+\.\d+)\s+'
    r'(?P<csa_b>-?\d+\.\d+)\s+(?P<csa_a>-?\d+\.\d+)\s+(?P<csa_n>-?\d+\.\d+)\s+'
    r'(?P<crsa_b>-?\d+\.\d+)\s+(?P<crsa_a>-?\d+\.\d+)\s+(?P<crsa_n>-?\d+\.\d+)\s+'
    r'(?P<dest>\S+)\s+(?P<from>\S+)\s+(?P<nmerge>\d+)\s+(?P<notes>.*?)\s*$'
)

QUAT_BRACKET_RE = re.compile(r'\[([^\]]*)\]')

THRESH_WALL_RE = re.compile(r'Wall\s+transition threshold\s*:\s*([\d.]+)')
THRESH_FLOOR_RE = re.compile(r'Floor instability threshold\s*:\s*([\d.]+)')
SUMQ_RE = re.compile(r'Sum of raw CSA weights.*?:\s*([\-\d.]+)')


def _parse_quat_list(quats_field: str) -> np.ndarray:
    """Parses a refQuat field that may contain one or more pipe-separated
    '[w x y z]' brackets (the new multi-quaternion symmetric-variant
    format) into a (k,4) array, one row per bracket, in the order they
    appear (matching the order of the comma-separated si/pose-index group
    on the same row when there's more than one).

    Also handles the old single-bracket format transparently (k == 1).
    """
    brackets = QUAT_BRACKET_RE.findall(quats_field)
    if not brackets:
        raise ValueError(f"Could not parse any '[w x y z]' quaternion from: {quats_field!r}")
    rows = []
    for b in brackets:
        vals = [float(v) for v in b.split()]
        if len(vals) != 4:
            raise ValueError(
                f"Expected 4 values (w x y z) inside quaternion bracket, got "
                f"{len(vals)}: {b!r}"
            )
        rows.append(vals)
    return np.array(rows, dtype=float)


@dataclass
class PoseEntry:
    si: int
    floor_plane: int
    theta_deg: float
    quat_wxyz: np.ndarray   # (k,4) array [w,x,y,z] -- k>=1 rows, one per
                             # symmetric-orientation variant parsed from the
                             # (possibly pipe-separated) refQuat field
    merged_ids: List[int]   # all pose indices in this row's si group, in
                             # the same order as quat_wxyz's rows (canonical
                             # pose_id == merged_ids[0])
    omega: float
    height: float
    ratio_wall: float
    ratio_floor: float
    csa_b: float
    csa_a: float
    csa_n: float
    crsa_b: float
    crsa_a: float
    crsa_n: float
    dest: Optional[int]
    received_from: Optional[int]
    n_merge: int
    notes: str

    @property
    def flag_transitions(self) -> bool:
        return "TRANSITIONS" in self.notes

    @property
    def flag_floor_unstable(self) -> bool:
        return "FLOOR-UNSTABLE" in self.notes

    @property
    def flag_zero_weight(self) -> bool:
        return self.notes.strip() not in ("", "-")


@dataclass
class ConditionData:
    part: str
    alpha: float          # actual physical roll used to tilt the chute
    beta: float            # actual physical pitch used to tilt the chute
    roll_tag: int           # MATLAB-rounded folder tag
    pitch_tag: int
    condition_dir: str
    folder_name: str
    thresh_wall: float
    thresh_floor: float
    sum_q: float
    poses: Dict[int, PoseEntry] = field(default_factory=dict)
    catalog: Optional[Dict[int, np.ndarray]] = None
    catalog_path: Optional[str] = None  # external poseCatalog.csv, if used
    geometric_path: Optional[str] = None  # *_GEOMETRIC_summary.txt, if used to build the catalog


def parse_master_summary(master_path: str) -> Tuple[Dict[int, PoseEntry], float, float, float]:
    poses: Dict[int, PoseEntry] = {}
    thresh_wall = 2.731   # ChutePoseAnalysis.m defaults, used if header parse fails
    thresh_floor = 2.296
    sum_q = 0.0

    with open(master_path, "r") as f:
        for line in f:
            m = THRESH_WALL_RE.search(line)
            if m:
                thresh_wall = float(m.group(1))
                continue
            m = THRESH_FLOOR_RE.search(line)
            if m:
                thresh_floor = float(m.group(1))
                continue
            m = SUMQ_RE.search(line)
            if m:
                sum_q = float(m.group(1))
                continue

            m = POSE_ROW_RE.match(line)
            if not m:
                continue
            g = m.groupdict()

            # si/plane/theta may each be a comma-separated group (new
            # multi-quaternion symmetric-variant format) or a single value
            # (old format, which is just a length-1 group). The canonical
            # pose_id is the FIRST id in the si group.
            si_group = [int(s) for s in g["si"].split(",")]
            si = si_group[0]
            plane_group = [int(v) for v in g["plane"].split(",")]

            quat_variants = _parse_quat_list(g["quats"])
            if len(quat_variants) != len(si_group):
                print(f"  [warn] {os.path.basename(master_path)}: pose row for "
                      f"si={g['si']} lists {len(si_group)} pose index/indices but "
                      f"{len(quat_variants)} quaternion(s) in refQuat -- these should "
                      f"match. Using all {len(quat_variants)} parsed quaternion(s) as "
                      f"symmetric variants for pose {si} regardless.")

            dest = None if g["dest"] == "-" else int(g["dest"])
            frm = None if g["from"] == "-" else int(g["from"])
            poses[si] = PoseEntry(
                si=si,
                merged_ids=si_group,
                floor_plane=plane_group[0],
                theta_deg=float(g["theta"].split(",")[0]),
                quat_wxyz=quat_variants,
                omega=float(g["omega"]),
                height=float(g["height"]),
                ratio_wall=float(g["rwall"]),
                ratio_floor=float(g["rfloor"]),
                csa_b=float(g["csa_b"]), csa_a=float(g["csa_a"]), csa_n=float(g["csa_n"]),
                crsa_b=float(g["crsa_b"]), crsa_a=float(g["crsa_a"]), crsa_n=float(g["crsa_n"]),
                dest=dest, received_from=frm, n_merge=int(g["nmerge"]), notes=g["notes"].strip(),
            )

    return poses, thresh_wall, thresh_floor, sum_q


# ===========================================================================
# Condition discovery
# ===========================================================================

def discover_conditions(parts_folder: str, alphas: List[float], betas: List[float],
                         parts_filter: Optional[List[str]],
                         results_root: Optional[str] = None,
                         dedupe_tol: float = 0.05) -> List[ConditionData]:
    if results_root is None:
        results_root = os.path.join(parts_folder, "results")
    if not os.path.isdir(results_root):
        raise FileNotFoundError(f"No results folder found at {results_root}")

    part_dirs = sorted(
        d for d in os.listdir(results_root)
        if os.path.isdir(os.path.join(results_root, d))
    )
    if parts_filter:
        part_dirs = [d for d in part_dirs if d in parts_filter]

    conditions: List[ConditionData] = []
    for part in part_dirs:
        for alpha in alphas:
            for beta in betas:
                roll_tag = matlab_round(alpha)
                pitch_tag = matlab_round(beta)
                folder_name = f"{part}_{roll_tag}R_{pitch_tag}P"
                condition_dir = os.path.join(results_root, part, folder_name)
                master_path = os.path.join(condition_dir, f"{folder_name}_MASTER_summary.txt")

                if not os.path.isfile(master_path):
                    print(f"  [skip] no MASTER_summary for {part} alpha={alpha} beta={beta} "
                          f"(expected {master_path})")
                    continue

                poses, thresh_wall, thresh_floor, sum_q = parse_master_summary(master_path)
                if not poses:
                    print(f"  [skip] {master_path} parsed but contained no stable poses")
                    continue

                cond = ConditionData(
                    part=part, alpha=alpha, beta=beta,
                    roll_tag=roll_tag, pitch_tag=pitch_tag,
                    condition_dir=condition_dir, folder_name=folder_name,
                    thresh_wall=thresh_wall, thresh_floor=thresh_floor, sum_q=sum_q,
                    poses=poses,
                )

                catalog_path = find_catalog_csv(condition_dir, folder_name)
                if catalog_path:
                    # An external poseCatalog.csv, if present, is still preferred
                    # (it's the authoritative exportPoseQuatCatalog.m output and
                    # may carry more precision / more variants than what fits on
                    # one MASTER_summary text line).
                    cond.catalog = load_pose_catalog(catalog_path)
                    cond.catalog_path = catalog_path
                else:
                    geom_path = find_geometric_summary(condition_dir, folder_name)
                    if geom_path:
                        # Second choice: build the catalog from the RAW-index
                        # quats parsed out of *_GEOMETRIC_summary.txt (this is
                        # where the multi-quaternion / symmetric-variant format
                        # actually lives -- MASTER_summary.txt keeps its own,
                        # separate CSA-weight-based merge grouping). For every
                        # MASTER PoseEntry, look up every raw index in its
                        # merged_ids group -- EXPANDED through the geometric
                        # file's own theta-group membership (raw_group), since
                        # MASTER may still list only ONE representative raw
                        # index per pose even when that index's geometric
                        # theta-group actually has more than one symmetric
                        # variant -- and stack whatever's found as that pose's
                        # symmetric-variant catalog entry.
                        raw_quats, raw_group = parse_geometric_summary(geom_path)
                        catalog: Dict[int, np.ndarray] = {}
                        missing_raw_ids = []
                        for si, pe in poses.items():
                            expanded_ids: List[int] = []
                            for rid in pe.merged_ids:
                                for gid in raw_group.get(rid, [rid]):
                                    if gid not in expanded_ids:
                                        expanded_ids.append(gid)

                            geom_found = [raw_quats[rid] for rid in expanded_ids if rid in raw_quats]
                            missing_raw_ids.extend(
                                (si, rid) for rid in expanded_ids if rid not in raw_quats
                            )
                            # MASTER's own parsed refQuat(s) are always kept -- they
                            # came straight off this pose's own row and are never
                            # wrong for this pose, unlike the GEOM cross-reference
                            # which can be incomplete. GEOM-derived variants are
                            # ADDED on top, not used as a replacement, and
                            # sign-invariant-deduped so the same orientation found
                            # via both sources doesn't inflate catalog[si].shape[0]
                            # against nMerge below.
                            variant_quats = _dedupe_quats(
                                list(pe.quat_wxyz) + geom_found,
                                tol=dedupe_tol,
                            )
                            catalog[si] = variant_quats
                        cond.catalog = catalog
                        cond.geometric_path = geom_path

                        if missing_raw_ids:
                            print(f"  [warn] {folder_name}: {len(missing_raw_ids)} raw pose "
                                  f"index/indices (after expanding MASTER_summary's merged "
                                  f"pose groups through each raw id's own geometric "
                                  f"theta-group) were not found in "
                                  f"{os.path.basename(geom_path)} (canonical_pose_id, "
                                  f"missing_raw_id): {missing_raw_ids} -- falling back to "
                                  f"MASTER's own refQuat for those specific variants.")

                        under_matched = [
                            (si, pe.n_merge, catalog[si].shape[0])
                            for si, pe in poses.items()
                            if pe.n_merge > catalog[si].shape[0]
                        ]
                        if under_matched:
                            print(f"  [warn] {folder_name}: even after cross-referencing "
                                  f"{os.path.basename(geom_path)} (including each raw id's "
                                  f"full theta-group), {len(under_matched)} pose(s) still "
                                  f"have fewer quaternion variants than their nMerge count "
                                  f"(pose_id, nMerge, quats_found): {under_matched} -- those "
                                  f"poses will still under-match (their remaining symmetric "
                                  f"variants must have been merged across theta-groups by "
                                  f"MASTER's own CSA-weight merge, which isn't recoverable "
                                  f"from GEOMETRIC_summary.txt alone -- a poseCatalog.csv "
                                  f"would be needed for those).")
                    else:
                        # Last resort: no external CSV and no GEOMETRIC_summary.txt
                        # found -- use the quaternion variant(s) already parsed
                        # directly from each MASTER pose row's own (possibly
                        # multi-bracket) refQuat field.
                        cond.catalog = {si: pe.quat_wxyz for si, pe in poses.items()}

                        under_matched = [
                            (si, pe.n_merge, pe.quat_wxyz.shape[0])
                            for si, pe in poses.items()
                            if pe.n_merge > pe.quat_wxyz.shape[0]
                        ]
                        if under_matched:
                            print(f"  [warn] no pose-catalog CSV or GEOMETRIC_summary.txt "
                                  f"found for {folder_name}; using the refQuat variant(s) "
                                  f"embedded in each MASTER_summary row, but "
                                  f"{len(under_matched)} pose(s) list fewer refQuat variants "
                                  f"than their nMerge count (pose_id, nMerge, quats_found): "
                                  f"{under_matched} -- those poses will still under-match.")

                conditions.append(cond)

    return conditions


# ===========================================================================
# Physics world
# ===========================================================================

def find_part_mesh(parts_folder: str, part_name: str) -> str:
    for ext in (".stl", ".obj", ".STL", ".OBJ"):
        candidate = os.path.join(parts_folder, part_name + ext)
        if os.path.isfile(candidate):
            return candidate
    matches = glob.glob(os.path.join(parts_folder, part_name + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find a mesh file for part '{part_name}' in {parts_folder}")


def describe_mesh_bounds(mesh_path: str, scale: float, label: str):
    """Returns (extents_scaled_m, ok)."""
    try:
        import trimesh
    except ImportError:
        print(f"  [scale-check] {label}: trimesh not installed, skipping size check "
              f"(pip install trimesh for this diagnostic)")
        return None, False

    try:
        mesh = trimesh.load(mesh_path, force="mesh")
    except Exception as e:
        print(f"  [scale-check] {label}: could not load '{mesh_path}' for sizing ({e})")
        return None, False

    raw_extents = np.asarray(mesh.extents, dtype=float)
    scaled_extents = raw_extents * scale

    print(f"  [scale-check] {label}: raw bounds size = "
          f"({raw_extents[0]:.4g}, {raw_extents[1]:.4g}, {raw_extents[2]:.4g})  "
          f"x scale={scale}  ->  real-world size (m) = "
          f"({scaled_extents[0]:.4g}, {scaled_extents[1]:.4g}, {scaled_extents[2]:.4g})")

    max_dim = float(np.max(scaled_extents))
    if max_dim > 5.0:
        print(f"    [!] WARNING: {label}'s largest scaled dimension is {max_dim:.3g} m "
              f"-- that's building-sized. If this mesh was authored in millimeters, "
              f"you likely need scale=0.001 (currently {scale}).")
    elif max_dim < 0.001:
        print(f"    [!] WARNING: {label}'s largest scaled dimension is {max_dim:.3g} m "
              f"-- smaller than a millimeter. Check whether `scale` is too aggressive.")

    return scaled_extents, True

def _read_raw_obj_vertices(obj_path: str) -> np.ndarray:
    """Parses 'v x y z' lines directly from an OBJ file, in file order,
    with NO vertex welding/deduplication/reordering. This preserves
    exactly the vertex numbering the mesh was authored with (OBJ vertex
    indices are 1-based in the file, e.g. 'v1' = the first 'v' line).

    trimesh.load(..., force="mesh") welds coincident vertices by default,
    which silently reindexes everything -- so a fixed vertex number like
    "6" can end up pointing at a totally different corner than the one
    the mesh was authored with, and that welding is exactly what broke
    the old index-based lookup here. Bypassing trimesh for this one read
    keeps the numbering stable and matches what you see when you open the
    .obj file in a text editor.
    """
    verts = []
    with open(obj_path, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("v ") or line == "v":
                parts = line.split()[1:4]
                if len(parts) == 3:
                    verts.append([float(v) for v in parts])
    return np.array(verts, dtype=float)

def compute_chute_drop_auto(chute_path: str, scale: float,
                             alpha_deg: float, beta_deg: float,
                             inset_frac: float = 0.1,
                             rim_vertex_ids: Tuple[int, int] = (1, 6)
                             ) -> Tuple[float, float, float]:
    """Reads chute mesh vertices at the given 1-based OBJ vertex numbers
    (default: v1 and v6, matching the authored mesh's own numbering --
    read directly from the raw file text, NOT trimesh's post-load
    `mesh.vertices`, since trimesh welds coincident vertices by default
    and that silently reorders/removes vertices), ROTATES them by this
    condition's actual chute tilt (alpha_deg roll, beta_deg pitch), and
    returns:
      - drop_x_auto: mean of their ROTATED X
      - y_min/y_max: their ROTATED Y range, inset by inset_frac of the
        range on each side

    IMPORTANT: because these are now POST-rotation coordinates, they must
    NOT be rotated a second time downstream. run_condition_trials()
    normally rotates drop_local_offset by chute_quat_xyzw before use --
    when --drop-y-auto is active, that second rotation must be skipped
    (see the matching change in run_condition_trials below), or the tilt
    gets double-applied and the drop point ends up nowhere near the
    chute.
    """
    if chute_path.lower().endswith(".obj"):
        verts_raw = _read_raw_obj_vertices(chute_path)
    else:
        import trimesh
        mesh = trimesh.load(chute_path, force="mesh")
        verts_raw = np.asarray(mesh.vertices, dtype=float)

    verts_local = verts_raw * scale

    id_a, id_b = rim_vertex_ids
    n = verts_local.shape[0]
    if id_a < 1 or id_a > n or id_b < 1 or id_b > n:
        raise ValueError(f"Chute mesh '{chute_path}' has {n} vertices -- "
                          f"requested vertex numbers {rim_vertex_ids} (1-based) "
                          f"are out of range.")

    idx = [id_a - 1, id_b - 1]

    chute_quat_xyzw = build_chute_quat_xyzw(alpha_deg, beta_deg)
    v_a_rot = rotate_vec_by_quat_xyzw(verts_local[idx[0]].tolist(), chute_quat_xyzw)
    v_b_rot = rotate_vec_by_quat_xyzw(verts_local[idx[1]].tolist(), chute_quat_xyzw)
    rotated = np.array([v_a_rot, v_b_rot])

    x_vals = rotated[:, 0]
    y_vals = rotated[:, 1]

    drop_x_auto = float(x_vals.mean())
    y_min, y_max = float(y_vals.min()), float(y_vals.max())
    y_range = y_max - y_min
    if y_range <= 1e-9:
        raise ValueError(f"Chute mesh '{chute_path}': vertices {rim_vertex_ids} "
                          f"have ~zero rotated Y extent at alpha={alpha_deg}, "
                          f"beta={beta_deg} -- can't compute a usable drop-y "
                          f"range for this tilt.")
    inset = y_range * inset_frac
    return drop_x_auto, y_min + inset, y_max - inset


def run_scale_sanity_check(chute_path: str, chute_scale: float,
                            stl_folder: str, conditions: List["ConditionData"],
                            part_scale: float):
    print("\nMesh scale sanity check (units: meters, after --part-scale / --chute-scale):")
    chute_extents, chute_ok = describe_mesh_bounds(chute_path, chute_scale, "chute")

    seen_parts = []
    for cond in conditions:
        if cond.part not in seen_parts:
            seen_parts.append(cond.part)

    for part in seen_parts:
        try:
            part_path = find_part_mesh(stl_folder, part)
        except FileNotFoundError as e:
            print(f"  [scale-check] {part}: {e}")
            continue
        part_extents, part_ok = describe_mesh_bounds(part_path, part_scale, part)

        if chute_ok and part_ok:
            chute_max = float(np.max(chute_extents))
            part_max = float(np.max(part_extents))
            if chute_max > 0 and part_max > 0:
                ratio = part_max / chute_max
                if ratio > 3.0:
                    print(f"    [!] WARNING: {part} ({part_max:.3g} m) is more than 3x "
                          f"larger than the chute ({chute_max:.3g} m). It will likely "
                          f"overshoot or clip through the chute mesh. Double check "
                          f"--part-scale / --chute-scale.")
                elif ratio < 0.02:
                    print(f"    [!] WARNING: {part} ({part_max:.3g} m) is less than 1/50th "
                          f"the size of the chute ({chute_max:.3g} m) -- it may be a "
                          f"near-invisible speck relative to the chute. Double check "
                          f"--part-scale / --chute-scale.")
    print("")


def ensure_obj_for_vhacd(mesh_path: str, cache_dir: str) -> str:
    """PyBullet's built-in VHACD only parses Wavefront OBJ text -- feeding it
    an STL will not raise a Python exception, it silently crashes/hangs the
    native VHACD code. Convert non-OBJ meshes to OBJ first."""
    ext = os.path.splitext(mesh_path)[1].lower()
    if ext == ".obj":
        return mesh_path

    try:
        import trimesh
    except ImportError as e:
        raise RuntimeError(
            f"'{mesh_path}' is a {ext} file but pybullet's VHACD only accepts "
            f".obj input. Install trimesh to auto-convert (`pip install trimesh`)."
        ) from e

    os.makedirs(cache_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(mesh_path))[0]
    obj_path = os.path.join(cache_dir, base + "_src.obj")
    if not os.path.isfile(obj_path):
        print(f"  Converting {os.path.basename(mesh_path)} -> OBJ for VHACD ...")
        mesh = trimesh.load(mesh_path, force="mesh")
        mesh.export(obj_path)
    return obj_path


def get_convex_decomposition(part_path: str, cache_dir: str) -> str:
    os.makedirs(cache_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(part_path))[0]
    out_path = os.path.join(cache_dir, base + "_vhacd.obj")
    log_path = os.path.join(cache_dir, base + "_vhacd_log.txt")
    if not os.path.isfile(out_path):
        vhacd_input_path = ensure_obj_for_vhacd(part_path, cache_dir)
        print(f"  Running VHACD convex decomposition for {base} ...")
        p.vhacd(vhacd_input_path, out_path, log_path,
                resolution=1000000, depth=20, maxNumVerticesPerCH=64)
    return out_path


def compute_part_center_of_mass(part_mesh_path: str, scale: float) -> Tuple[float, float, float]:
    """Computes the part's TRUE center of mass (assuming uniform density),
    scaled the same way the part will be scaled for simulation.
    """
    try:
        import trimesh
    except ImportError:
        print(f"  [warn] trimesh not installed -- cannot compute true center of mass for "
              f"'{part_mesh_path}'. Falling back to (0,0,0), i.e. assuming the mesh's own "
              f"file origin is its centroid, which is very likely WRONG for a CAD-exported "
              f"part and can produce spurious stable edge/corner poses. "
              f"Install trimesh (`pip install trimesh`) to fix this.")
        return (0.0, 0.0, 0.0)

    try:
        mesh = trimesh.load(part_mesh_path, force="mesh")
        if not mesh.is_watertight:
            print(f"  [warn] '{part_mesh_path}' is not watertight -- center_mass may be "
                  f"approximate. Consider repairing the mesh (trimesh.repair.fill_holes) if "
                  f"this part continues to show implausible resting poses.")
        com = np.asarray(mesh.center_mass, dtype=float) * scale
        print(f"  [com-check] {os.path.basename(part_mesh_path)}: center of mass (post-scale) "
              f"= ({com[0]:.4g}, {com[1]:.4g}, {com[2]:.4g}) m")
        return (float(com[0]), float(com[1]), float(com[2]))
    except Exception as e:
        print(f"  [warn] could not compute center of mass for '{part_mesh_path}' ({e}). "
              f"Falling back to (0,0,0) -- see warning above about why this can be wrong.")
        return (0.0, 0.0, 0.0)


def reset_condition_world(chute_path: str, chute_scale, chute_concave: bool,
                           roll_deg: float, pitch_deg: float, catch_plane: bool,
                           z_lift: float = 2.0, gui: bool = False,
                           chute_color=(0.7, 0.7, 0.75, 1.0)):
    """Rebuilds the world for one (roll, pitch) condition."""
    p.resetSimulation()
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)

    p.setTimeStep(1.0 / 240.0)
    p.setPhysicsEngineParameter(numSubSteps=10, numSolverIterations=150,
                                 contactBreakingThreshold=0.0001)

    if catch_plane:
        p.loadURDF("plane.urdf")

    chute_quat_xyzw = build_chute_quat_xyzw(roll_deg, pitch_deg)

    flags = (p.GEOM_FORCE_CONCAVE_TRIMESH | p.GEOM_CONCAVE_INTERNAL_EDGE) if chute_concave else 0
    chute_collision = p.createCollisionShape(
        p.GEOM_MESH, fileName=chute_path, meshScale=chute_scale, flags=flags
    )
    chute_visual = p.createVisualShape(
        p.GEOM_MESH, fileName=chute_path, meshScale=chute_scale, rgbaColor=list(chute_color)
    )
    chute_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=chute_collision,
        baseVisualShapeIndex=chute_visual,
        basePosition=[0, 0, z_lift],
        baseOrientation=chute_quat_xyzw,
    )

    if gui:
        aabb_min, aabb_max = p.getAABB(chute_id)
        aabb_min = np.array(aabb_min)
        aabb_max = np.array(aabb_max)
        center = ((aabb_min + aabb_max) / 2.0).tolist()
        diag = float(np.linalg.norm(aabb_max - aabb_min))
        cam_dist = max(diag * 0.25, 0.5)
        p.resetDebugVisualizerCamera(
            cameraDistance=cam_dist, cameraYaw=35, cameraPitch=-35,
            cameraTargetPosition=[2.5, -0.2, z_lift],
        )
        print(f"    [gui] chute bounding box: min={aabb_min.tolist()} max={aabb_max.tolist()} "
              f"-> camera target={center}, distance={cam_dist:.3g}")

    return chute_id, chute_quat_xyzw


def spawn_part(part_collision_path: str, part_scale, mass: float, drop_pos, orientation,
               part_color=(0.9, 0.3, 0.2, 1.0), part_com=(0.0, 0.0, 0.0)):
    col = p.createCollisionShape(p.GEOM_MESH, fileName=part_collision_path, meshScale=part_scale)
    vis = p.createVisualShape(p.GEOM_MESH, fileName=part_collision_path, meshScale=part_scale,
                               rgbaColor=list(part_color))
    body_id = p.createMultiBody(
        baseMass=mass,
        baseCollisionShapeIndex=col,
        baseVisualShapeIndex=vis,
        basePosition=drop_pos,
        baseOrientation=orientation,
        baseInertialFramePosition=list(part_com),
    )

    aabb_min, aabb_max = p.getAABB(body_id)
    part_size = float(np.min(np.array(aabb_max) - np.array(aabb_min)))
    if part_size > 0:
        try:
            p.changeDynamics(body_id, -1,
                              ccdSweptSphereRadius=part_size * 0.2,
                              ccdMotionThreshold=part_size * 0.2)
        except TypeError:
            p.changeDynamics(body_id, -1, ccdSweptSphereRadius=part_size * 0.2)

    return body_id


GRAVITY_MAG = 9.81


def apply_random_perturbation(body_id, weight: float, force_frac: float, torque_frac: float,
                               rng: np.random.Generator):
    theta = rng.uniform(0, 2 * np.pi)
    force = [force_frac * weight * np.cos(theta), force_frac * weight * np.sin(theta), 0.0]
    torque_dir = rng.normal(size=3)
    torque_dir /= (np.linalg.norm(torque_dir) + 1e-12)
    torque = (torque_frac * weight * torque_dir).tolist()
    p.applyExternalForce(body_id, -1, forceObj=force, posObj=[0, 0, 0], flags=p.WORLD_FRAME)
    p.applyExternalTorque(body_id, -1, torqueObj=torque, flags=p.WORLD_FRAME)


# ===========================================================================
# Tunable surface-roughness (micro-texture skitter) model
# ===========================================================================
#
# See the "SURFACE ROUGHNESS MODEL" block in the module docstring for the
# full rationale. Short version: this is a time-correlated random-force
# process applied while the part is sliding/rolling in contact with the
# chute, standing in for surface texture/perforation grip without needing
# to model the texture geometrically. It is independent of, and layered on
# top of, the existing per-trial friction/restitution redraw and the
# ambient_jitter false-settle guard.

@dataclass
class RoughnessModel:
    enabled: bool = False
    force_frac: float = 0.02        # lateral noise force, fraction of part weight
    torque_frac: float = 0.02       # noise torque, fraction of part weight (moment-ish scale)
    correlation_steps: int = 8      # sim steps one sampled "bump" direction is held before resampling
    velocity_gate: float = 0.002    # below this speed (m/s), part is treated as at-rest -> no noise
    velocity_scale_ref: float = 0.3 # speed (m/s) at which noise reaches full amplitude
    contact_only: bool = True       # only perturb while touching the chute


def _sample_bump(rng: np.random.Generator, force_frac: float, torque_frac: float, weight: float):
    theta = rng.uniform(0, 2 * np.pi)
    force = np.array([np.cos(theta), np.sin(theta), 0.0]) * force_frac * weight
    torque_dir = rng.normal(size=3)
    torque_dir /= (np.linalg.norm(torque_dir) + 1e-12)
    torque = torque_dir * torque_frac * weight
    return force, torque


def apply_roughness_step(body_id, weight: float, rough: RoughnessModel,
                          rng: np.random.Generator, state: dict):
    """Call once per stepSimulation() while settling, if roughness is enabled.

    `state` is a small persistent dict the caller keeps across the whole
    run_until_settled() call, e.g. {"steps_left": 0, "force": None, "torque": None}
    -- it holds the currently-sampled "bump" so it persists across
    correlation_steps consecutive calls instead of resampling every step.
    """
    if not rough.enabled or weight <= 0:
        return

    if rough.contact_only:
        contacts = p.getContactPoints(bodyA=body_id)
        if not contacts:
            return

    lin_vel, _ = p.getBaseVelocity(body_id)
    speed = float(np.linalg.norm(lin_vel))
    if speed < rough.velocity_gate:
        return  # essentially at rest -- a stationary part doesn't feel rolling texture

    scale = min(speed / rough.velocity_scale_ref, 1.0)

    if state.get("steps_left", 0) <= 0:
        force, torque = _sample_bump(rng, rough.force_frac, rough.torque_frac, weight)
        state["force"], state["torque"] = force, torque
        state["steps_left"] = rough.correlation_steps
    state["steps_left"] -= 1

    p.applyExternalForce(body_id, -1, forceObj=(state["force"] * scale).tolist(),
                          posObj=[0, 0, 0], flags=p.WORLD_FRAME)
    p.applyExternalTorque(body_id, -1, torqueObj=(state["torque"] * scale).tolist(),
                           flags=p.WORLD_FRAME)


def run_until_settled(body_id, max_steps=2400, lin_thresh=0.005, ang_thresh=0.02,
                       stable_steps_required=60, gui=False, sleep=0.0,
                       ambient_jitter=False, weight=0.0, jitter_interval=200,
                       jitter_force_frac=0.02, jitter_torque_frac=0.02, rng=None,
                       roughness: Optional[RoughnessModel] = None):
    stable_count = 0
    rough_state = {"steps_left": 0, "force": None, "torque": None}
    for step in range(max_steps):
        if ambient_jitter and weight > 0 and rng is not None and step > 0 and step % jitter_interval == 0:
            apply_random_perturbation(body_id, weight, jitter_force_frac, jitter_torque_frac, rng)
            stable_count = 0
        if roughness is not None and rng is not None:
            apply_roughness_step(body_id, weight, roughness, rng, rough_state)
        p.stepSimulation()
        if gui and sleep > 0:
            time.sleep(sleep)
        lin_vel, ang_vel = p.getBaseVelocity(body_id)
        lin_speed = np.linalg.norm(lin_vel)
        ang_speed = np.linalg.norm(ang_vel)
        if lin_speed < lin_thresh and ang_speed < ang_thresh:
            stable_count += 1
            if stable_count >= stable_steps_required:
                return step, True
        else:
            stable_count = 0
    return max_steps, False


def verify_settled_not_knife_edge(body_id, mass: float, rng: np.random.Generator,
                                   nudge_force_frac=0.3, nudge_torque_frac=0.3,
                                   nudge_steps=10, recheck_steps=200,
                                   gui=False, sleep=0.0,
                                   roughness: Optional[RoughnessModel] = None):
    weight = mass * GRAVITY_MAG
    for _ in range(nudge_steps):
        apply_random_perturbation(body_id, weight, nudge_force_frac, nudge_torque_frac, rng)
        p.stepSimulation()
        if gui and sleep > 0:
            time.sleep(sleep)
    _, still_settled = run_until_settled(body_id, max_steps=recheck_steps,
                                          gui=gui, sleep=sleep,
                                          weight=weight, rng=rng, roughness=roughness)
    return still_settled


# ===========================================================================
# PTFE contact-dynamics sampling
# ===========================================================================

@dataclass
class FrictionModel:
    chute_friction_min: float = 0.05
    chute_friction_max: float = 0.1
    part_friction_min: float = 0.05
    part_friction_max: float = 0.1
    chute_restitution_min: float = 0.01
    chute_restitution_max: float = 0.03
    part_restitution_min: float = 0.01
    part_restitution_max: float = 0.03
    snag_prob: float = 0.0005
    snag_friction_boost: float = 4.0
    rolling_friction: float = 0.005
    spinning_friction: float = 0.005
    chute_linear_damping: float = 0.04
    chute_angular_damping: float = 0.1
    part_linear_damping: float = 0.05
    part_angular_damping: float = 0.1

    def sample(self, rng: np.random.Generator):
        chute_fric = rng.uniform(self.chute_friction_min, self.chute_friction_max)
        part_fric = rng.uniform(self.part_friction_min, self.part_friction_max)
        chute_rest = rng.uniform(self.chute_restitution_min, self.chute_restitution_max)
        part_rest = rng.uniform(self.part_restitution_min, self.part_restitution_max)
        snagged = rng.random() < self.snag_prob
        if snagged:
            chute_fric = min(chute_fric * self.snag_friction_boost, 0.9)
        return chute_fric, part_fric, chute_rest, part_rest, snagged


# ===========================================================================
# Per-condition Monte Carlo trial loop
# ===========================================================================

def run_condition_trials(cond: ConditionData, chute_id, chute_quat_xyzw: Tuple[float, float, float, float],
                          part_collision_path: str, args, friction_model: FrictionModel,
                          rng: np.random.Generator, out_dir: str,
                          part_com: Tuple[float, float, float] = (0.0, 0.0, 0.0),
                          roughness: Optional[RoughnessModel] = None):

    trial_rows = []
    counts: Dict[Optional[int], int] = {}
    n_unsettled = 0
    n_snagged = 0
    n_knife_edge_rejected = 0

    z_lift = args.chute_z_lift

    # NOTE: the drop position offset used to be computed ONCE here, before
    # the trial loop, since it was fixed for the whole condition. It is now
    # computed fresh INSIDE the loop every trial, because the Y component
    # can be randomized per-trial (see --drop-y-min/--drop-y-max below) --
    # see the drop-position block at the top of the trial loop.

    for trial in range(args.n):
        chute_fric, part_fric, chute_rest, part_rest, snagged = friction_model.sample(rng)
        if snagged:
            n_snagged += 1
        p.changeDynamics(chute_id, -1, lateralFriction=chute_fric, restitution=chute_rest,
                          rollingFriction=friction_model.rolling_friction,
                          spinningFriction=friction_model.spinning_friction,
                          linearDamping=friction_model.chute_linear_damping,
                          angularDamping=friction_model.chute_angular_damping)

        # --- Per-trial drop position -----------------------------------
        # Y is randomized uniformly in [--drop-y-min, --drop-y-max] when
        # BOTH flags are given; otherwise the fixed --drop-xy Y value is
        # used every trial (original behavior, unchanged). X always comes
        # from --drop-xy[0], and drop height always from --drop-height --
        # only Y supports randomization currently.
        if args.drop_y_min is not None and args.drop_y_max is not None:
            drop_y = rng.uniform(args.drop_y_min, args.drop_y_max)
        else:
            drop_y = args.drop_xy[1]

        drop_local_offset = [args.drop_xy[0], drop_y, args.drop_height]
        if args.drop_y_auto:
            # X/Y here already came out of compute_chute_drop_auto POST-rotation
            # (v1/v6 rotated by this condition's tilt) -- do NOT rotate again,
            # or the tilt gets double-applied and the part drops nowhere near
            # the chute. Height still needs to be added in world Z, same as
            # the non-auto path.
            drop_pos = [drop_local_offset[0], drop_local_offset[1],
                        drop_local_offset[2] + z_lift]
        else:
            drop_pos_offset = rotate_vec_by_quat_xyzw(drop_local_offset, chute_quat_xyzw)
            drop_pos = [drop_pos_offset[0], drop_pos_offset[1], drop_pos_offset[2] + z_lift]

        orn0 = random_quaternion_xyzw(rng)

        body_id = spawn_part(part_collision_path, [args.part_scale] * 3, args.part_mass,
                              drop_pos, orn0, part_color=args.part_color, part_com=part_com)
        p.changeDynamics(body_id, -1, lateralFriction=part_fric, restitution=part_rest,
                          rollingFriction=friction_model.rolling_friction,
                          spinningFriction=friction_model.spinning_friction,
                          linearDamping=friction_model.part_linear_damping,
                          angularDamping=friction_model.part_angular_damping)

        part_weight = args.part_mass * GRAVITY_MAG
        steps, settled = run_until_settled(
            body_id, max_steps=args.max_steps, gui=args.gui, sleep=args.sleep,
            ambient_jitter=not args.no_ambient_jitter, weight=part_weight,
            jitter_interval=args.ambient_jitter_interval,
            jitter_force_frac=args.ambient_jitter_force_frac,
            jitter_torque_frac=args.ambient_jitter_torque_frac, rng=rng,
            roughness=roughness)

        knife_edge_rejected = False
        if settled:
            # Confirm this isn't a knife-edge/point-balance false positive
            # (see verify_settled_not_knife_edge docstring) before trusting
            # the pose match below.
            still_settled = verify_settled_not_knife_edge(
                body_id, mass=args.part_mass, rng=rng,
                nudge_force_frac=args.nudge_force_frac, nudge_torque_frac=args.nudge_torque_frac,
                nudge_steps=args.nudge_steps, recheck_steps=min(200, args.max_steps),
                gui=args.gui, sleep=args.sleep, roughness=roughness)
            if not still_settled:
                knife_edge_rejected = True
                settled = False
                n_knife_edge_rejected += 1

        if not settled:
            n_unsettled += 1

        final_pos, final_orn_world = p.getBasePositionAndOrientation(body_id)
        final_orn_local = rotate_to_chute_local(final_orn_world, chute_quat_xyzw)
        matched_id, match_dist = match_pose_in_catalog(final_orn_local, cond.catalog,
                                                         args.quat_match_tol)

        counts[matched_id] = counts.get(matched_id, 0) + 1

        pe = cond.poses.get(matched_id) if matched_id is not None else None

        # Report position relative to the chute's local origin (Z=0 at the
        # floor/wall seam), i.e. subtract the world-space z_lift back out.
        final_x_rel = final_pos[0]
        final_y_rel = final_pos[1]
        final_z_rel = final_pos[2] - z_lift

        trial_rows.append({
            "trial": trial,
            # drop_x/drop_y/drop_height record the ACTUAL local-frame drop
            # position used for this trial -- drop_y varies trial-to-trial
            # when --drop-y-min/--drop-y-max are active, so this is what
            # you'd want to correlate against outcome (settled/matched_pose)
            # if you're checking whether entry position affects the result.
            "drop_x": args.drop_xy[0], "drop_y": drop_y, "drop_height": args.drop_height,
            "init_qx": orn0[0], "init_qy": orn0[1], "init_qz": orn0[2], "init_qw": orn0[3],
            "final_x": final_x_rel, "final_y": final_y_rel, "final_z": final_z_rel,
            "final_qx_local": final_orn_local[0], "final_qy_local": final_orn_local[1],
            "final_qz_local": final_orn_local[2], "final_qw_local": final_orn_local[3],
            "settled": settled, "steps_to_settle": steps,
            "knife_edge_rejected": knife_edge_rejected,
            "matched_pose_id": matched_id if matched_id is not None else -1,
            "match_dist_rad": match_dist,
            "geom_notes": pe.notes if pe else "",
            "geom_ratio_wall": pe.ratio_wall if pe else "",
            "geom_ratio_floor": pe.ratio_floor if pe else "",
            "chute_friction": chute_fric, "part_friction": part_fric,
            "chute_restitution": chute_rest, "part_restitution": part_rest,
            "snagged": snagged,
        })

        p.removeBody(body_id)

        if trial % 100 == 0:
            print(f"    trial {trial}/{args.n}  settled={settled}  "
                  f"matched_pose={matched_id}  dist={match_dist:.4f}")

    os.makedirs(os.path.join(out_dir, cond.part, cond.folder_name), exist_ok=True)
    trials_csv = os.path.join(out_dir, cond.part, cond.folder_name, f"{cond.folder_name}_trials.csv")
    with open(trials_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(trial_rows[0].keys()))
        writer.writeheader()
        writer.writerows(trial_rows)

    print(f"    -> wrote {len(trial_rows)} trials to {trials_csv}")
    print(f"    -> unsettled: {n_unsettled}/{args.n}   snagged: {n_snagged}/{args.n}   "
          f"knife-edge rejected: {n_knife_edge_rejected}/{args.n}")

    return counts, n_unsettled


def write_condition_summary(cond: ConditionData, counts: Dict[Optional[int], int], out_dir: str):
    total = sum(v for k, v in counts.items() if k is not None)
    rows = []
    unstable_hits = []

    all_pose_ids = sorted(set(cond.poses.keys()) | {k for k in counts.keys() if k is not None})
    for pose_id in all_pose_ids:
        pe = cond.poses.get(pose_id)
        sim_count = counts.get(pose_id, 0)
        sim_freq = 100.0 * sim_count / total if total else 0.0
        row = {
            "part": cond.part, "alpha_deg": cond.alpha, "beta_deg": cond.beta,
            "pose_id": pose_id,
            "sim_count": sim_count, "sim_freq_pct": sim_freq,
            "geom_CSA_B_pct": pe.csa_b if pe else "",
            "geom_CSA_A_pct": pe.csa_a if pe else "",
            "geom_CSA_N_pct": pe.csa_n if pe else "",
            "geom_CRSA_B_pct": pe.crsa_b if pe else "",
            "geom_CRSA_A_pct": pe.crsa_a if pe else "",
            "geom_CRSA_N_pct": pe.crsa_n if pe else "",
            "ratio_wall": pe.ratio_wall if pe else "",
            "thresh_wall": cond.thresh_wall,
            "ratio_floor": pe.ratio_floor if pe else "",
            "thresh_floor": cond.thresh_floor,
            "geom_notes": pe.notes if pe else "",
            "flagged_unstable_by_geometry": bool(pe and (pe.flag_transitions or pe.flag_floor_unstable)),
        }
        rows.append(row)
        if row["flagged_unstable_by_geometry"] and sim_count > 0:
            unstable_hits.append((pose_id, sim_count))

    # unmatched trials
    unmatched = counts.get(None, 0)
    if unmatched:
        rows.append({
            "part": cond.part, "alpha_deg": cond.alpha, "beta_deg": cond.beta,
            "pose_id": -1, "sim_count": unmatched,
            "sim_freq_pct": 100.0 * unmatched / total if total else 0.0,
            "geom_CSA_B_pct": "", "geom_CSA_A_pct": "", "geom_CSA_N_pct": "",
            "geom_CRSA_B_pct": "", "geom_CRSA_A_pct": "", "geom_CRSA_N_pct": "",
            "ratio_wall": "", "thresh_wall": cond.thresh_wall,
            "ratio_floor": "", "thresh_floor": cond.thresh_floor,
            "geom_notes": "UNMATCHED (no catalog pose within quat_match_tol)",
            "flagged_unstable_by_geometry": False,
        })

    summary_csv = os.path.join(out_dir, cond.part, cond.folder_name,
                                f"{cond.folder_name}_sim_summary.csv")
    with open(summary_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"    -> wrote condition summary to {summary_csv}")
    if unstable_hits:
        print(f"    [!] simulation matched poses the geometric analysis flagged as "
              f"unstable/transitioning: {unstable_hits}")

    return rows


# ===========================================================================
# Main
# ===========================================================================

def build_arg_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(description=__doc__,
                                  formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--parts-folder", required=True,
                     help="Base folder. Used as the default location for both STL meshes "
                          "and the 'results/' subfolder, unless --stl-folder / --results-root "
                          "are given to point at different locations for each.")
    ap.add_argument("--stl-folder", default=None,
                     help="Folder containing the part STL/OBJ mesh files, if different from "
                          "--parts-folder (e.g. STLs live in a separate subfolder).")
    ap.add_argument("--results-root", default=None,
                     help="Folder that directly contains the per-part result folders "
                          "(<results-root>/PartName/PartName_RxxPxx/..._MASTER_summary.txt), "
                          "for layouts that don't wrap results in a 'results/' subfolder. "
                          "Defaults to '<parts-folder>/results' if not given.")
    ap.add_argument("--chute", required=True, help="Path to chute mesh (.obj/.stl)")
    ap.add_argument("--chute-scale", type=float, default=1.0,
                     help="Uniform scale applied to the chute mesh to convert it to meters "
                          "(PyBullet is SI). E.g. if chute.obj was authored in millimeters, "
                          "pass 0.001. Check the mesh scale sanity-check printout at startup "
                          "if unsure.")
    ap.add_argument("--chute-concave", action="store_true")
    ap.add_argument("--chute-z-lift", type=float, default=2.0,
                     help="World +Z offset (meters, post-scale) the chute is placed at before "
                          "being rotated by roll/pitch. Default 2.0 m is safely larger than "
                          "this chute's own dimensions at any reasonable tilt; increase if "
                          "you use a larger chute mesh or very steep angles. The drop point "
                          "and reported final_x/y/z are automatically adjusted to stay "
                          "consistent with this lift.")
    ap.add_argument("--no-catch-plane", action="store_true",
                     help="Disable the flat plane.urdf floor beneath the chute")
    ap.add_argument("--chute-color", type=float, nargs=4, default=[0.7, 0.7, 0.75, 1.0],
                     help="Chute RGBA color, each 0-1 (default: light gray)")
    ap.add_argument("--part-color", type=float, nargs=4, default=[0.9, 0.3, 0.2, 1.0],
                     help="Part RGBA color, each 0-1 (default: reddish)")

    ap.add_argument("--parts", default=None,
                     help="Comma-separated subset of part names to run (default: all found)")
    ap.add_argument("--alphas", default="15,17.5,20",
                     help="Comma-separated roll angles (deg)")
    ap.add_argument("--betas", default="15,25,35,45",
                     help="Comma-separated pitch angles (deg)")

    ap.add_argument("--n", type=int, default=1000, help="Drop trials per (part, angle) condition")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default="batch_drop_results")

    ap.add_argument("--part-mass", type=float, default=0.01)
    ap.add_argument("--part-scale", type=float, default=1.0,
                     help="Uniform scale applied to part meshes to convert them to meters "
                          "(PyBullet is SI). STL exports are very commonly millimeters -- "
                          "if so, pass 0.001. Check the mesh scale sanity-check printout at "
                          "startup if unsure; an unscaled mm part will be ~1000x too large.")
    ap.add_argument("--drop-height", type=float, default=0.3,
                     help="Drop height (m) above chute origin, along the chute's local +Z")
    ap.add_argument("--drop-xy", type=float, nargs=2, default=[0.0, 0.0],
                     help="Drop XY offset (m) in the chute's local frame. If --drop-y-min "
                          "and --drop-y-max are BOTH given, the Y value here is ignored and "
                          "Y is instead randomized per trial in [drop-y-min, drop-y-max]; "
                          "the X value here is always used as-is.")
    ap.add_argument("--drop-y-min", type=float, default=None,
                     help="If set together with --drop-y-max, the drop offset's Y component "
                          "is drawn fresh from Uniform(drop-y-min, drop-y-max) EVERY TRIAL, "
                          "instead of using the fixed --drop-xy Y value for the whole "
                          "condition. Models dispersion in where parts actually enter the "
                          "chute. Both --drop-y-min and --drop-y-max must be given together "
                          "for randomization to activate; giving only one is silently "
                          "ignored (falls back to the fixed --drop-xy Y value) rather than "
                          "raising an error.")
    ap.add_argument("--drop-y-max", type=float, default=None,
                     help="See --drop-y-min.")
    ap.add_argument("--drop-y-auto", action="store_true",
                     help="Automatically compute drop_x and drop_y bounds PER CONDITION "
                          "from the two chute mesh vertices that end up highest in Z after "
                          "that condition's roll/pitch tilt (the entry rim). Sets drop_x to "
                          "their local-X mean and drop_y_min/max to their local-Y range, "
                          "inset by --drop-y-auto-inset-frac. Overrides --drop-xy's X value "
                          "and any manually passed --drop-y-min/--drop-y-max.")
    ap.add_argument("--drop-y-auto-inset-frac", type=float, default=0.1,
                     help="Fraction of the two-rim-vertex Y range to inset from each edge "
                          "when --drop-y-auto is used. Default 0.1 (10%%).")
    ap.add_argument("--quat-match-tol", type=float, default=0.05,
                     help="Geodesic distance (rad) tolerance for pose matching; "
                          "should match ChutePoseAnalysis.m's quatMatchTol (default 0.05)")
    ap.add_argument("--max-steps", type=int, default=2400)

    ap.add_argument("--gui", action="store_true")
    ap.add_argument("--sleep", type=float, default=0.0)
    ap.add_argument("--dry-run", action="store_true",
                     help="Only discover and print conditions; do not simulate")
    ap.add_argument("--skip-scale-check", action="store_true",
                     help="Skip the automatic mesh real-world-size sanity check at startup")

    ap.add_argument("--chute-friction-min", type=float, default=0.05)
    ap.add_argument("--chute-friction-max", type=float, default=0.1)
    ap.add_argument("--part-friction-min", type=float, default=0.05)
    ap.add_argument("--part-friction-max", type=float, default=0.1)
    ap.add_argument("--chute-restitution-min", type=float, default=0.01)
    ap.add_argument("--chute-restitution-max", type=float, default=0.03)
    ap.add_argument("--part-restitution-min", type=float, default=0.01)
    ap.add_argument("--part-restitution-max", type=float, default=0.03)
    ap.add_argument("--snag-prob", type=float, default=0.0005)
    ap.add_argument("--snag-friction-boost", type=float, default=2.0)
    ap.add_argument("--rolling-friction", type=float, default=0.005)
    ap.add_argument("--spinning-friction", type=float, default=0.005)

    ap.add_argument("--chute-linear-damping", type=float, default=0.04)
    ap.add_argument("--chute-angular-damping", type=float, default=0.1)
    ap.add_argument("--part-linear-damping", type=float, default=0.05)
    ap.add_argument("--part-angular-damping", type=float, default=0.1)

    ap.add_argument("--nudge-force-frac", type=float, default=0.3,
                     help="Post-settle knife-edge check: lateral force per step, as a "
                          "fraction of the part's own weight (mass * g). Default 0.3.")
    ap.add_argument("--nudge-torque-frac", type=float, default=0.3,
                     help="Post-settle knife-edge check: random-axis torque per step, as a "
                          "fraction of the part's own weight. Default 0.3.")
    ap.add_argument("--nudge-steps", type=int, default=10,
                     help="Number of consecutive steps the knife-edge nudge force/torque is "
                          "re-applied for. Default 10.")
    ap.add_argument("--no-ambient-jitter", action="store_true",
                     help="Disable periodic ambient force/torque perturbation during "
                          "run_until_settled.")
    ap.add_argument("--ambient-jitter-interval", type=int, default=200,
                     help="Steps between ambient jitter perturbations. Default 200.")
    ap.add_argument("--ambient-jitter-force-frac", type=float, default=0.02,
                     help="Ambient jitter lateral force, as a fraction of the part's own "
                          "weight. Default 0.02.")
    ap.add_argument("--ambient-jitter-torque-frac", type=float, default=0.02,
                     help="Ambient jitter random-axis torque, as a fraction of the part's "
                          "own weight. Default 0.02.")

    ap.add_argument("--roughness-enabled", action="store_true",
                     help="Enable the tunable surface-roughness (micro-texture skitter) "
                          "model: a time-correlated random lateral force/torque applied "
                          "while the part is sliding/rolling in contact with the chute. "
                          "See the SURFACE ROUGHNESS MODEL section in this script's module "
                          "docstring for the full rationale. Off by default (identical "
                          "behavior to before this flag existed).")
    ap.add_argument("--roughness-force-frac", type=float, default=0.02,
                     help="Roughness lateral noise force, as a fraction of the part's own "
                          "weight. Higher = grippier/rougher perceived texture. Default 0.02.")
    ap.add_argument("--roughness-torque-frac", type=float, default=0.02,
                     help="Roughness random-axis noise torque, as a fraction of the part's "
                          "own weight. Default 0.02.")
    ap.add_argument("--roughness-correlation-steps", type=int, default=8,
                     help="Number of consecutive simulation steps one sampled 'bump' "
                          "direction is held before resampling. At dt=1/240s, 8 steps ~= "
                          "33ms per bump. Larger = coarser texture (e.g. modeling "
                          "perforation-hole spacing), smaller = finer grit. Default 8.")
    ap.add_argument("--roughness-velocity-gate", type=float, default=0.002,
                     help="Below this linear speed (m/s) the part is treated as at-rest and "
                          "roughness is not applied, so it can't perpetually buzz a settled "
                          "part awake. Default 0.002.")
    ap.add_argument("--roughness-velocity-scale-ref", type=float, default=0.3,
                     help="Linear speed (m/s) at which roughness noise reaches full "
                          "amplitude; scales linearly (capped at 1.0) below that. Set near "
                          "the part's typical sliding speed on your real chute. Default 0.3.")
    ap.add_argument("--roughness-no-contact-only", action="store_true",
                     help="By default roughness only applies while the part is touching the "
                          "chute (so free-fall / entry dispersion is undisturbed). Pass this "
                          "to apply it regardless of contact state.")
    return ap


def main(argv: Optional[List[str]] = None):
    ap = build_arg_parser()

    if argv is None:
        if _running_under_notebook_kernel():
            if NOTEBOOK_ARGV is None:
                print(
                    "No CLI arguments detected and no NOTEBOOK_ARGV set.\n"
                    "You're running inside a Jupyter/IPython kernel, so this "
                    "script can't read --parts-folder/--chute from sys.argv.\n"
                    "Set them before calling main(), e.g.:\n\n"
                    "    import chute_drop_batch as cdb\n"
                    "    cdb.NOTEBOOK_ARGV = [\n"
                    "        '--parts-folder', r'C:\\path\\to\\partsFolder',\n"
                    "        '--chute', r'C:\\path\\to\\chute.obj',\n"
                    "        '--chute-concave', '--part-scale', '0.001',\n"
                    "        '--n', '1000',\n"
                    "        '--out-dir', 'batch_drop_results',\n"
                    "    ]\n"
                    "    cdb.main()\n"
                )
                return
            argv = NOTEBOOK_ARGV
        else:
            argv = sys.argv[1:]

    args = ap.parse_args(argv)

    

    alphas = [float(a) for a in args.alphas.split(",")]
    betas = [float(b) for b in args.betas.split(",")]
    parts_filter = [s.strip() for s in args.parts.split(",")] if args.parts else None

    stl_folder = args.stl_folder if args.stl_folder else args.parts_folder

    print("Discovering conditions ...")
    conditions = discover_conditions(args.parts_folder, alphas, betas, parts_filter,
                                      results_root=args.results_root,
                                      dedupe_tol=args.quat_match_tol)
    print(f"Found {len(conditions)} runnable (part, alpha, beta) condition(s):")
    for c in conditions:
        if c.catalog_path:
            catalog_src = f"catalog CSV={os.path.basename(c.catalog_path)}"
        elif c.geometric_path:
            catalog_src = f"catalog from GEOMETRIC_summary={os.path.basename(c.geometric_path)}"
        else:
            catalog_src = "catalog from MASTER_summary refQuat only (no external source found)"
        print(f"  {c.part}: alpha={c.alpha} beta={c.beta} -> {c.folder_name} "
              f"({len(c.poses)} stable poses, {catalog_src})")

    if not conditions:
        return

    if not args.skip_scale_check:
        run_scale_sanity_check(args.chute, args.chute_scale, stl_folder, conditions, args.part_scale)

    if args.dry_run:
        return

    friction_model = FrictionModel(
        chute_friction_min=args.chute_friction_min, chute_friction_max=args.chute_friction_max,
        part_friction_min=args.part_friction_min, part_friction_max=args.part_friction_max,
        chute_restitution_min=args.chute_restitution_min, chute_restitution_max=args.chute_restitution_max,
        part_restitution_min=args.part_restitution_min, part_restitution_max=args.part_restitution_max,
        snag_prob=args.snag_prob, snag_friction_boost=args.snag_friction_boost,
        rolling_friction=args.rolling_friction, spinning_friction=args.spinning_friction,
        chute_linear_damping=args.chute_linear_damping, chute_angular_damping=args.chute_angular_damping,
        part_linear_damping=args.part_linear_damping, part_angular_damping=args.part_angular_damping,
    )

    roughness_model = RoughnessModel(
        enabled=args.roughness_enabled,
        force_frac=args.roughness_force_frac,
        torque_frac=args.roughness_torque_frac,
        correlation_steps=args.roughness_correlation_steps,
        velocity_gate=args.roughness_velocity_gate,
        velocity_scale_ref=args.roughness_velocity_scale_ref,
        contact_only=not args.roughness_no_contact_only,
    )
    if roughness_model.enabled:
        print(f"Surface roughness model ENABLED: force_frac={roughness_model.force_frac} "
              f"torque_frac={roughness_model.torque_frac} "
              f"correlation_steps={roughness_model.correlation_steps} "
              f"velocity_gate={roughness_model.velocity_gate} "
              f"velocity_scale_ref={roughness_model.velocity_scale_ref} "
              f"contact_only={roughness_model.contact_only}")

    rng = np.random.default_rng(args.seed)
    mode = p.GUI if args.gui else p.DIRECT
    p.connect(mode)

    os.makedirs(args.out_dir, exist_ok=True)
    vhacd_cache_dir = os.path.join(args.out_dir, "_vhacd_cache")

    combined_rows = []
    part_convex_cache: Dict[str, str] = {}
    part_com_cache: Dict[str, Tuple[float, float, float]] = {}

    for cond in conditions:
        print(f"\n=== {cond.part}  alpha={cond.alpha}  beta={cond.beta}  "
              f"({cond.folder_name}) ===")

        if args.drop_y_auto:
            drop_x_auto, y_min, y_max = compute_chute_drop_auto(
                args.chute, args.chute_scale, cond.alpha, cond.beta,
                args.drop_y_auto_inset_frac)
            args.drop_xy[0] = drop_x_auto
            args.drop_y_min, args.drop_y_max = y_min, y_max
            print(f"  [drop-y-auto] alpha={cond.alpha} beta={cond.beta} -> "
                  f"drop_x_auto={drop_x_auto:.4g}  drop_y_min={y_min:.4g}  "
                  f"drop_y_max={y_max:.4g}")
        elif (args.drop_y_min is None) != (args.drop_y_max is None):
            print("  [warn] only one of --drop-y-min/--drop-y-max was given -- Y drop "
                  "randomization needs BOTH, so it will NOT activate; falling back to the "
                  "fixed --drop-xy Y value for every trial.")

        if cond.part not in part_convex_cache:
            part_mesh_path = find_part_mesh(stl_folder, cond.part)
            part_convex_cache[cond.part] = get_convex_decomposition(part_mesh_path, vhacd_cache_dir)
            part_com_cache[cond.part] = compute_part_center_of_mass(part_mesh_path, args.part_scale)
        part_collision_path = part_convex_cache[cond.part]
        part_com = part_com_cache[cond.part]

        chute_id, chute_quat_xyzw = reset_condition_world(
            args.chute, [args.chute_scale] * 3, args.chute_concave,
            cond.alpha, cond.beta, catch_plane=not args.no_catch_plane,
            z_lift=args.chute_z_lift, gui=args.gui, chute_color=args.chute_color,
        )

        counts, n_unsettled = run_condition_trials(
            cond, chute_id, chute_quat_xyzw, part_collision_path,
            args, friction_model, rng, args.out_dir, part_com=part_com,
            roughness=roughness_model,
        )

        combined_rows.extend(write_condition_summary(cond, counts, args.out_dir))

    p.disconnect()

    combined_csv = os.path.join(args.out_dir, "combined_summary.csv")
    if combined_rows:
        with open(combined_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(combined_rows[0].keys()))
            writer.writeheader()
            writer.writerows(combined_rows)
        print(f"\nDone. Combined summary across all conditions: {combined_csv}")


if __name__ == "__main__":
    main()

No CLI arguments detected and no NOTEBOOK_ARGV set.
You're running inside a Jupyter/IPython kernel, so this script can't read --parts-folder/--chute from sys.argv.
Set them before calling main(), e.g.:

    import chute_drop_batch as cdb
    cdb.NOTEBOOK_ARGV = [
        '--parts-folder', r'C:\path\to\partsFolder',
        '--chute', r'C:\path\to\chute.obj',
        '--chute-concave', '--part-scale', '0.001',
        '--n', '1000',
        '--out-dir', 'batch_drop_results',
    ]
    cdb.main()



In [12]:
NOTEBOOK_ARGV = [
    "--parts-folder", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet",
    "--stl-folder", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM",
    "--results-root", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet",
    "--chute", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM\chute.obj",
    "--chute-concave",
    "--parts", "Df4a,Ql4i,Rk2i",
    "--alphas", "15,25,35,45",
    "--betas", "15,18,20",
    "--chute-restitution-min", "0.5",
    "--chute-restitution-max", "0.6",
    "--part-restitution-min", "0.5",
    "--part-restitution-max", "0.6",
    "--chute-friction-min", "0",
    "--chute-friction-max", "0",
    "--part-friction-min", "0",
    "--part-friction-max", "0",
    "--drop-height", "8",
    "--drop-y-auto",
    "--drop-y-auto-inset-frac", "0.15",
    "--chute-z-lift", "5",
    "--n", "2000",
    "--part-scale", "0.025",
    "--max-steps", "1200",
    "--out-dir", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\batch_drop_results_full",
]
main()

Discovering conditions ...
  [warn] Rk2i_15R_15P_MASTER_summary.txt: pose row for si=1 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 1 regardless.
  [warn] Rk2i_15R_15P_MASTER_summary.txt: pose row for si=2 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 2 regardless.
  [warn] Rk2i_15R_15P_MASTER_summary.txt: pose row for si=3 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 3 regardless.
  [warn] Rk2i_15R_15P_MASTER_summary.txt: pose row for si=4 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 4 regardless.
  [warn] Rk2i_15R_15P_MASTER_summary.txt: pose row for si=5 lists 1 pose index/indices but 2 quat

In [7]:
NOTEBOOK_ARGV = [
    "--parts-folder", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet",
    "--stl-folder", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM",
    "--results-root", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet",
    "--chute", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM\chute.obj",
    "--chute-concave",
    "--alphas", "35",
    "--betas", "20",
    "--chute-restitution-min", "0.5",
    "--chute-restitution-max", "0.6",
    "--part-restitution-min", "0.5",
    "--part-restitution-max", "0.6",
    "--chute-friction-min", "0",
    "--chute-friction-max", "0",
    "--part-friction-min", "0",
    "--part-friction-max", "0",
    "--drop-height", "11",
    "--drop-y-auto",
    "--drop-y-auto-inset-frac", "0.15",
    "--chute-z-lift", "5",
    "--n", "25",
    "--gui",
    "--sleep", "0.0025",
    "--part-scale", "0.025",
    "--max-steps", "1200",
    "--parts", "Rk2i",
    "--out-dir", r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\batch_drop_results_gui_test",
]
main()

# note: find the top two vertices x,y and set the y min-max based on that. it seems to be dropping on the back wall, concentrating most of the values on p8/10

Discovering conditions ...
  [warn] Rk2i_35R_20P_MASTER_summary.txt: pose row for si=1 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 1 regardless.
  [warn] Rk2i_35R_20P_MASTER_summary.txt: pose row for si=2 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 2 regardless.
  [warn] Rk2i_35R_20P_MASTER_summary.txt: pose row for si=3 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 3 regardless.
  [warn] Rk2i_35R_20P_MASTER_summary.txt: pose row for si=4 lists 1 pose index/indices but 2 quaternion(s) in refQuat -- these should match. Using all 2 parsed quaternion(s) as symmetric variants for pose 4 regardless.
  [warn] Rk2i_35R_20P_MASTER_summary.txt: pose row for si=5 lists 1 pose index/indices but 2 quat

error: Not connected to physics server.

In [ ]:
# ============================================================
# One-shot quaternion probe -- run AFTER the cell that defines
# chute_drop_batch's functions (reset_condition_world, spawn_part,
# run_until_settled, etc.) -- uses them directly, no import needed.
# ============================================================

import numpy as np
import pybullet as p

# --- CONFIG -- edit these ---
PART_MESH_PATH = r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM\Ql4i.stl"
CHUTE_PATH     = r"C:\Users\benny\OneDrive\Documents\LLM\PyBullet\SIM\chute.obj"
PART_SCALE     = 0.01
CHUTE_SCALE    = 1.0
CHUTE_CONCAVE  = True
ROLL_DEG  = 25.0
PITCH_DEG = 20.0
DROP_HEIGHT = 2.0
DROP_XY     = [-0.75, -1.25]
PART_MASS   = 0.01
MAX_STEPS   = 20000
SLEEP       = 0.025
VHACD_CACHE_DIR = "quat_probe_vhacd_cache"

rng = np.random.default_rng(None)

try:
    p.disconnect()
except Exception:
    pass
p.connect(p.GUI)

chute_id, chute_quat_xyzw = reset_condition_world(
    CHUTE_PATH, [CHUTE_SCALE] * 3, CHUTE_CONCAVE,
    ROLL_DEG, PITCH_DEG, catch_plane=True, z_lift=2.0, gui=True,
)
p.changeDynamics(chute_id, -1, lateralFriction=0.08, restitution=0.02,
                  rollingFriction=0.005, spinningFriction=0.005,
                  linearDamping=0.04, angularDamping=0.1)

part_collision_path = get_convex_decomposition(PART_MESH_PATH, VHACD_CACHE_DIR)
part_com = compute_part_center_of_mass(PART_MESH_PATH, PART_SCALE)

drop_local_offset = [DROP_XY[0], DROP_XY[1], DROP_HEIGHT]
drop_pos_offset = rotate_vec_by_quat_xyzw(drop_local_offset, chute_quat_xyzw)
drop_pos = [drop_pos_offset[0], drop_pos_offset[1], drop_pos_offset[2] + 2.0]

orn0 = random_quaternion_xyzw(rng)

body_id = spawn_part(part_collision_path, [PART_SCALE] * 3, PART_MASS,
                      drop_pos, orn0, part_com=part_com)
p.changeDynamics(body_id, -1, lateralFriction=0.08, restitution=0.02,
                  rollingFriction=0.005, spinningFriction=0.005,
                  linearDamping=0.05, angularDamping=0.1)

print(f"Dropping at roll={ROLL_DEG} pitch={PITCH_DEG}, init orn (xyzw) = {orn0}")

steps, settled = run_until_settled(
    body_id, max_steps=MAX_STEPS, gui=True, sleep=SLEEP,
    ambient_jitter=True, weight=PART_MASS * GRAVITY_MAG,
    jitter_interval=200, jitter_force_frac=0.02, jitter_torque_frac=0.02, rng=rng)

still_settled = True
if settled:
    still_settled = verify_settled_not_knife_edge(
        body_id, mass=PART_MASS, rng=rng,
        nudge_force_frac=0.3, nudge_torque_frac=0.3,
        nudge_steps=10, recheck_steps=200, gui=True, sleep=SLEEP)

final_pos, final_orn_world = p.getBasePositionAndOrientation(body_id)
final_orn_local = rotate_to_chute_local(final_orn_world, chute_quat_xyzw)
x, y, z, w = final_orn_local

print("=" * 70)
print(f"settled           : {settled}")
print(f"knife-edge check  : {'passed' if still_settled else 'FAILED (still moving after nudge)'}")
print(f"steps to settle   : {steps}")
print(f"final world pos   : {final_pos}")
print(f"final LOCAL orn (xyzw) : {final_orn_local}")
print(f"final LOCAL orn (wxyz) : ({w:.4f}, {x:.4f}, {y:.4f}, {z:.4f})")
print("=" * 70)

NameError: name 'reset_condition_world' is not defined